In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

print('Sube tu labels.yolov8.zip...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
EXTRACT_DIR = Path('/content/dataset')
EXTRACT_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(EXTRACT_DIR)

print('\nContenido extraido:')
!find /content/dataset -type f | sort

In [ ]:
from pathlib import Path
import shutil, yaml

DATASET_DIR = Path('/content/dataset')

src_images = DATASET_DIR / 'train' / 'images'
src_labels = DATASET_DIR / 'train' / 'labels'
dst_images = DATASET_DIR / 'images' / 'train'
dst_labels = DATASET_DIR / 'labels' / 'train'

dst_images.mkdir(parents=True, exist_ok=True)
dst_labels.mkdir(parents=True, exist_ok=True)

for f in src_images.glob('*'):
    shutil.copy2(f, dst_images / f.name)

for f in src_labels.glob('*.txt'):
    shutil.copy2(f, dst_labels / f.name)

imgs = list(dst_images.glob('*'))
lbls = list(dst_labels.glob('*.txt'))
print(f'Imagenes : {len(imgs)}')
print(f'Labels   : {len(lbls)}')

YAML_PATH = DATASET_DIR / 'dataset_animales.yaml'
config = {
    'path': str(DATASET_DIR),
    'train': 'images/train',
    'val':   'images/train',
    'test':  'images/train',
    'nc':    1,
    'names': ['animal_exotico'],
}
with open(YAML_PATH, 'w') as f:
    yaml.dump(config, f, sort_keys=False)

print(f'\nYAML listo: {YAML_PATH}')
!cat /content/dataset/dataset_animales.yaml

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

model.train(
    data     = '/content/dataset/dataset_animales.yaml',
    imgsz    = 640,
    epochs   = 50,
    batch    = 16,
    patience = 20,
    device   = "cpu",
    project  = '/content/runs',
    name     = 'animales_v1',
    exist_ok = True,
    save     = True,
    hsv_h    = 0.015,
    hsv_s    = 0.7,
    degrees  = 10.0,
    flipud   = 0.0,
    fliplr   = 0.5,
    mosaic   = 1.0,
)

print('Entrenamiento completo.')